# PIPELINE MLOPS WINE QUALITY PREDICTION
Dalam pengembangan sistem MLOps untuk Wine Quality Prediction dengan pendekatan secara end-to-end machine learning, digunakan dua framework utama yaitu TensorFlow dan TensorFlow Extended (TFX). TensorFlow digunakan untuk membangun, melatih, dan mengevaluasi arsitektur model deep learning yang akan memprediksi kualitas wine. Sementara itu, TensorFlow Extended (TFX) digunakan sebagai platform produksi untuk mengotomatisasi seluruh siklus data dan model mulai dari data ingestion, validasi skema, transformasi fitur, hingga manajemen perilisan model ke tahap produksi secara konsisten dan terukur.

## Pipeline Environment Setup & Variable Initialization
Tahap ini menginisialisasi jalur (path) dan variabel lingkungan yang digunakan oleh **TensorFlow Extended (TFX)** untuk mengelola *artifact* dan *metadata* produksi:
* **PIPELINE_NAME:** Identitas unik untuk pipeline produksi (`setyana-dev-pipeline`).
* **PIPELINE_ROOT:** Direktori penyimpanan output *artifact* (data `TFRecord`, grafik `Transform`, checkpoint `Trainer`, dan evaluasi `TFMA`).
* **METADATA_PATH:** Lokasi file SQLite (`metadata.sqlite`) untuk menyimpan *data lineage* dan riwayat eksekusi komponen melalui ML Metadata (MLMD).
* **SERVING_MODEL_DIR:** Direktori ekspor untuk *SavedModel* terbaik yang lolos validasi untuk di-mount ke TensorFlow Serving.
* **DATA_ROOT:** Direktori asal untuk memuat dataset mentah `winequality-red.csv`.
* **BeamDagRunner:** Menggunakan orchestrator berbasis Apache Beam untuk mengeksekusi seluruh komponen pipeline secara otomatis dan terstruktur via skrip `local_pipeline.py`.


## Komponen Pipeline TFX yang Dieksekusi

Seluruh tahapan siklus data dan model diatur secara modular di dalam direktori `modules/` dan dieksekusi secara berurutan oleh **BeamDagRunner** di dalam skrip `local_pipeline.py`:

1. **CsvExampleGen:** Melakukan *ingest* data mentah dari format CSV menjadi `TFRecord` serta membaginya menjadi data latih (*train*) dan evaluasi (*eval*).
2. **StatisticsGen & SchemaGen:** Menganalisis parameter statistik deskriptif dan membuat skema data otomatis untuk mendefinisikan batasan fitur.
3. **ExampleValidator:** Memvalidasi dataset terhadap skema untuk mendeteksi adanya data *drift* atau anomali.
4. **Transform:** Melakukan pra-pemrosesan data secara konsisten (seperti *Z-Score scaling*) menggunakan modul `winequality_transform.py`.
5. **Tuner & Trainer:** Mencari kombinasi *hyperparameter* optimal dan melatih arsitektur model DNN melalui modul `winequality_trainer.py`.
6. **Resolver & Evaluator:** Memeriksa dan membandingkan performa model baru terhadap *baseline model* menggunakan kriteria *threshold* akurasi (TFMA).
7. **Pusher:** Mengekspor *SavedModel* terbaik ke folder tujuan serving (`serving_model/`) jika model berhasil mendapatkan status *blessed*.


## Orkestrasi Pipeline 
Berkas `local_pipeline.py` merupakan skrip orkestrator utama yang mengintegrasikan seluruh komponen TFX di atas. Skrip ini menggunakan **BeamDagRunner** sebagai mesin pengeksekusi lokal untuk menjalankan pipeline secara otomatis, mencatat riwayat ke ML Metadata (`metadata.sqlite`), dan mengekspor model akhir secara langsung ke dalam folder `output/`.

In [1]:
%run local_pipeline.py

2026-09-14 13:32:38.568463: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-14 13:32:38.675774: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory
2026-09-14 13:32:38.675791: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.
2026-09-14 13:32:38.697681: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-09-14 13:32:39.425655: W tensorflow/stream_executor/platform/de

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying winequality_transform.py -> build/lib
copying components.py -> build/lib
copying winequality_trainer.py -> build/lib
installing to /tmp/tmpe_5fex6_
running install
running install_lib
copying build/lib/winequality_transform.py -> /tmp/tmpe_5fex6_
copying build/lib/components.py -> /tmp/tmpe_5fex6_
copying build/lib/winequality_trainer.py -> /tmp/tmpe_5fex6_
running install_egg_info
running egg_info
creating tfx_user_code_Transform.egg-info
writing tfx_user_code_Transform.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Transform.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Transform.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Transform.egg-info/SOURCES.txt'
Copying tfx_user_code_Transform.egg-info to /tmp/t

/home/indra/miniconda3/envs/wine-tfx/lib/python3.9/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
  warn(
/home/indra/miniconda3/envs/wine-tfx/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
INFO:absl:Successfully built user code wheel distribution 

running bdist_wheel
running build
running build_py
creating build
creating build/lib
copying winequality_transform.py -> build/lib
copying components.py -> build/lib
copying winequality_trainer.py -> build/lib
installing to /tmp/tmp4b0av36b
running install
running install_lib
copying build/lib/winequality_transform.py -> /tmp/tmp4b0av36b
copying build/lib/components.py -> /tmp/tmp4b0av36b
copying build/lib/winequality_trainer.py -> /tmp/tmp4b0av36b
running install_egg_info
running egg_info
creating tfx_user_code_Tuner.egg-info
writing tfx_user_code_Tuner.egg-info/PKG-INFO
writing dependency_links to tfx_user_code_Tuner.egg-info/dependency_links.txt
writing top-level names to tfx_user_code_Tuner.egg-info/top_level.txt
writing manifest file 'tfx_user_code_Tuner.egg-info/SOURCES.txt'
reading manifest file 'tfx_user_code_Tuner.egg-info/SOURCES.txt'
writing manifest file 'tfx_user_code_Tuner.egg-info/SOURCES.txt'
Copying tfx_user_code_Tuner.egg-info to /tmp/tmp4b0av36b/tfx_user_code_Tuner-0

/home/indra/miniconda3/envs/wine-tfx/lib/python3.9/site-packages/wheel/bdist_wheel.py:4: FutureWarning: The 'wheel' package is no longer the canonical location of the 'bdist_wheel' command, and will be removed in a future release. Please update to setuptools v70.1 or later which contains an integrated version of this command.
  warn(
/home/indra/miniconda3/envs/wine-tfx/lib/python3.9/site-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ********************************************************************************

!!
  self.initialize_options()
INFO:absl:Successfully built user code wheel distribution 

...

INFO:absl:Using output/setyana-dev-pipeline/Trainer/model/8/Format-Serving as baseline model.


INFO:absl:The 'example_splits' parameter is not set, using 'eval' split.
INFO:absl:Evaluating model.
INFO:absl:udf_utils.get_fn {'fairness_indicator_thresholds': 'null', 'eval_config': '{\n  "metrics_specs": [\n    {\n      "metrics": [\n        {\n          "class_name": "ExampleCount"\n        },\n        {\n          "class_name": "SparseCategoricalAccuracy",\n          "threshold": {\n            "change_threshold": {\n              "absolute": 0.0001,\n              "direction": "HIGHER_IS_BETTER"\n            },\n            "value_threshold": {\n              "lower_bound": 0.3\n            }\n          }\n        }\n      ]\n    }\n  ],\n  "model_specs": [\n    {\n      "label_key": "quality_xf"\n    }\n  ],\n  "slicing_specs": [\n    {}\n  ]\n}', 'example_splits': 'null'} 'custom_extractors'


INFO:absl:Evaluation complete. Results written to output/setyana-dev-pipeline/Evaluator/evaluation/28.
INFO:absl:Checking validation results.


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`
INFO:absl:Blessing result False written to output/setyana-dev-pipeline/Evaluator/blessing/28.
INFO:absl:Cleaning up stateless execution info.
INFO:absl:Execution 28 succeeded.
INFO:absl:Cleaning up stateful execution info.
INFO:absl:Publishing output artifacts defaultdict(<class 'list'>, {'evaluation': [Artifact(artifact: uri: "output/setyana-dev-pipeline/Evaluator/evaluation/28"
, artifact_type: name: "ModelEvaluation"
)], 'blessing': [Artifact(artifact: uri: "output/setyana-dev-pipeline/Evaluator/blessing/28"
, artifact_type: name: "ModelBlessing"
)]}) for execution 28
INFO:absl:MetadataStore with DB connection initialized
INFO:absl:node Evaluator is finished.
INFO:absl:node Pusher is running.
INFO:absl:Running launcher for node_info {
  type {
    name: "tfx.components.pusher.component.Pusher"
    base_type: DEPLOY
  }
  id: "Pusher"
}
contexts {
  contexts {
    type {
      name: "pipeline"
    }


### Visualisasi Parameter Statistik (StatisticsGen)
Memuat InteractiveContext yang diarahkan ke folder *pipeline root* (`output/setyana-dev-pipeline`) untuk mengakses komponen *artifact* yang telah dihasilkan oleh `BeamDagRunner`. Menampilkan visualisasi statistik deskriptif dari dataset (data *train* dan *eval*) menggunakan TensorFlow Data Validation (TFDV) untuk memeriksa distribusi data, nilai minimum/maksimum, serta mendeteksi kehilangan data (*missing values*).

In [46]:
import os

import tensorflow_data_validation as tfdv

stats_base = 'output/setyana-dev-pipeline/StatisticsGen/statistics/'
latest_folder = max(os.listdir(stats_base), key=int)
stats_uri = os.path.join(stats_base, latest_folder)
train_stats_path = os.path.join(stats_uri, 'Split-train', 'FeatureStats.pb')
eval_stats_path = os.path.join(stats_uri, 'Split-eval', 'FeatureStats.pb')

if not os.path.exists(train_stats_path):
    train_stats_path = os.path.join(stats_uri, 'split-train', 'FeatureStats.pb')
    eval_stats_path = os.path.join(stats_uri, 'split-eval', 'FeatureStats.pb')

print(f"Load Statistics: {stats_uri}")

train_stats = tfdv.load_stats_binary(train_stats_path)
eval_stats = tfdv.load_stats_binary(eval_stats_path)

tfdv.visualize_statistics(lhs_statistics=train_stats, rhs_statistics=eval_stats,
                          lhs_name='TRAIN_DATASET', rhs_name='EVAL_DATASET')




Load Statistics: output/setyana-dev-pipeline/StatisticsGen/statistics/3


### Visualisasi Skema Data (SchemaGen)
Menampilkan tabel skema data hasil ekstraksi otomatis. Skema ini mendefinisikan tipe data dari setiap fitur fisikokimia anggur, mendeteksi nama fitur berspasi, serta menentukan batasan nilai yang valid untuk mendeteksi *data drift*.


In [ ]:
import os

import tensorflow_data_validation as tfdv

schema_base = 'output/setyana-dev-pipeline/SchemaGen/schema/'
latest_schema_folder = max(os.listdir(schema_base), key=int)
schema_uri = os.path.join(schema_base, latest_schema_folder)

print(f"Load Schema: {schema_uri}")

schema_file_path = os.path.join(schema_uri, 'schema.pbtxt')
schema = tfdv.load_schema_text(schema_file_path)

tfdv.display_schema(schema)



🔄 Memuat skema secara dinamis dari: output/setyana-dev-pipeline/SchemaGen/schema/4


,Type,Presence,Valency,Domain
Feature name,,,,
'alcohol',FLOAT,required,,-
'chlorides',FLOAT,required,,-
'citric acid',FLOAT,required,,-
'density',FLOAT,required,,-
'fixed acidity',FLOAT,required,,-
'free sulfur dioxide',FLOAT,required,,-
'pH',FLOAT,required,,-
'quality',INT,required,,-
'residual sugar',FLOAT,required,,-


### Analisis Performa Model (Evaluator)
Menampilkan visualisasi hasil evaluasi model mendalam menggunakan TensorFlow Model Analysis (TFMA). Tahap ini memverifikasi nilai metrik `SparseCategoricalAccuracy` terhadap batas *threshold* untuk menentukan apakah model layak mendapatkan status *blessed* sebelum dirilis.


In [ ]:
import os

import tensorflow_model_analysis as tfma

eval_base = 'output/setyana-dev-pipeline/Evaluator/evaluation/'
latest_eval_folder = max(os.listdir(eval_base), key=int)
eval_uri = os.path.join(eval_base, latest_eval_folder)

print(f"Load Evaluation: {eval_uri}")

tfma_result = tfma.load_eval_result(eval_uri)
tfma.view.render_slicing_metrics(tfma_result)

metrics_gen = tfma.load_metrics(eval_uri)

for metrics_per_slice in metrics_gen:
    for metric_entry in metrics_per_slice.metric_keys_and_values:
        metric_name = metric_entry.key.name
        metric_value = metric_entry.value.double_value.value
        print(f"{metric_name}: {round(metric_value, 4)}")


Memuat hasil evaluasi secara dinamis dari: output/setyana-dev-pipeline/Evaluator/evaluation/28

=== RINGKASAN METRIK EVALUASI MODEL (EKSPLISIT) ===
📈 sparse_categorical_accuracy: 0.5285
📈 loss: 1.058
📈 sparse_categorical_accuracy: 0.5285
📈 loss: 1.058
📈 sparse_categorical_accuracy: 0.5285
📈 sparse_categorical_accuracy: 0.5285
📈 example_count: 333.0
📈 example_count: 333.0
📈 sparse_categorical_accuracy: 0.0
📈 loss: 0.0
📈 sparse_categorical_accuracy: 0.0
📈 example_count: 0.0


## Testing

### Model Inference Testing (Uji Coba API)
Tahap ini merupakan demonstrasi pengujian model yang telah berhasil dideploy ke lingkungan server produksi menggunakan REST API. Data sampel fitur fisikokimia anggur dikemas ke dalam format standar `tf.train.Example`, dikodekan ke dalam bentuk string `base64`, lalu dikirimkan ke server menggunakan metode HTTP POST. Proses pengujian ini bertujuan memastikan bahwa *Inference Server* mampu merespons permintaan prediksi secara deterministik dan *real-time*.


In [14]:
import base64
import requests
import tensorflow as tf

sample_example = tf.train.Example(
    features=tf.train.Features(
        feature={
            'fixed acidity': tf.train.Feature(float_list=tf.train.FloatList(value=[12.0])), # Kunci: Sangat asam
            'volatile acidity': tf.train.Feature(float_list=tf.train.FloatList(value=[1.20])),
            'citric acid': tf.train.Feature(float_list=tf.train.FloatList(value=[0.00])),
            'residual sugar': tf.train.Feature(float_list=tf.train.FloatList(value=[1.2])),
            'chlorides': tf.train.Feature(float_list=tf.train.FloatList(value=[0.20])),
            'free sulfur dioxide': tf.train.Feature(float_list=tf.train.FloatList(value=[5.0])),
            'total sulfur dioxide': tf.train.Feature(float_list=tf.train.FloatList(value=[15.0])),
            'density': tf.train.Feature(float_list=tf.train.FloatList(value=[1.0020])),
            'pH': tf.train.Feature(float_list=tf.train.FloatList(value=[3.80])),
            'sulphates': tf.train.Feature(float_list=tf.train.FloatList(value=[0.30])),
            'alcohol': tf.train.Feature(float_list=tf.train.FloatList(value=[8.0])) # Kunci: Alkohol sangat rendah
        }
    )
)

serialized_example = sample_example.SerializeToString()
b64_example = base64.b64encode(serialized_example).decode('utf-8')

endpoint = 'http://localhost:8501/v1/models/wine-quality-model:predict'
payload = {
    'instances': [
        {'b64': b64_example}
    ]
}

response = requests.post(endpoint, json=payload)

print('Status:', response.status_code)
print('Prediction Response:', response.json())

Status: 200
Prediction Response: {'predictions': [[1.32794248e-05, 0.803120434, 0.190737456, 0.00612855284, 2.57224428e-07, 7.00030366e-13]]}
